# 03 — Drift Simulation

Streams the held-out test set through the champion model, monitors residuals with
ADWIN / Page-Hinkley / KSWIN, and demonstrates what happens when drift fires.
A synthetic step-shift injection at the end guarantees at least one high-severity event.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.stream import stream_test_data
from src.drift.monitor import DriftMonitor, DriftEvent
from src.drift.retrainer import DriftRetrainer
from src.models.registry import load_champion, get_champion_rmse

MODEL_NAME = 'driftpilot-forecaster'
TARGET_COL = 'OT'
FEATURE_COLS = None  # filled after first row

## 1. Load Champion Model & Val Split

In [ ]:
champion = load_champion(MODEL_NAME)
val_df   = pd.read_parquet('../data/processed/val.parquet')
test_df  = pd.read_parquet('../data/processed/test.parquet')

print(f'Champion loaded. Current val RMSE: {get_champion_rmse(MODEL_NAME):.4f}')
print(f'Test split: {len(test_df):,} rows')

## 2. Real-Data Simulation Loop

In [ ]:
monitor   = DriftMonitor()
retrainer = DriftRetrainer(val_df=val_df, model_name=MODEL_NAME)

actuals, predictions, residuals = [], [], []
drift_events: list[DriftEvent] = []

for row in stream_test_data(delay_seconds=0.0):
    y_true = row[TARGET_COL]
    if FEATURE_COLS is None:
        FEATURE_COLS = [k for k in row if k != TARGET_COL]

    x_df   = pd.DataFrame([{k: row[k] for k in FEATURE_COLS}])
    y_pred = float(champion.predict(x_df)[0])

    actuals.append(y_true)
    predictions.append(y_pred)
    residuals.append(y_true - y_pred)

    retrainer.ingest(row)
    event = monitor.update(y_pred=y_pred, y_true=y_true)
    if event:
        drift_events.append(event)

print(f'Stream complete: {monitor.row_index:,} rows processed')
print(f'Drift events detected: {monitor.drift_count}')
for ev in drift_events:
    print(f'  row {ev.row_index:4d} | {ev.severity:4s} | detectors={ev.triggered_detectors}')

## 3. Forecast vs Actual — with Drift Markers

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=actuals,     name='Actual OT',    line=dict(color='steelblue')))
fig.add_trace(go.Scatter(y=predictions, name='Predicted OT', line=dict(color='orange', dash='dot')))

for ev in drift_events:
    color = 'red' if ev.severity == 'high' else 'coral'
    fig.add_vline(x=ev.row_index, line_color=color, line_dash='dash',
                  annotation_text=ev.severity, annotation_position='top right')

fig.update_layout(
    title='OT Forecast vs Actual (test set) — Drift Events Marked',
    xaxis_title='Test row index',
    yaxis_title='Oil Temperature',
    hovermode='x unified',
    height=450,
)
fig.show()

## 4. Residual Trace

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=residuals, name='Residual (actual - pred)',
                          line=dict(color='grey'), opacity=0.7))
fig2.add_hline(y=0, line_dash='dot', line_color='black')

for ev in drift_events:
    color = 'red' if ev.severity == 'high' else 'coral'
    fig2.add_vline(x=ev.row_index, line_color=color, line_dash='dash')

fig2.update_layout(
    title='Residuals over Time',
    xaxis_title='Test row index',
    yaxis_title='Residual',
    height=350,
)
fig2.show()

## 5. Synthetic Drift Injection

Inject a +5 step-shift on `OT_lag_1` starting at row 200.
ADWIN and PageHinkley should both fire within ~50 rows, triggering a high-severity event.

In [ ]:
from src.drift.monitor import DriftMonitor

synth_monitor = DriftMonitor()
synth_events = []
synth_actuals, synth_preds, synth_residuals = [], [], []

SHIFT_START = 200
SHIFT_MAGNITUDE = 5.0

for i, row in enumerate(stream_test_data(delay_seconds=0.0)):
    row = dict(row)  # make mutable copy

    # Inject step-shift after row 200
    if i >= SHIFT_START:
        row['OT_lag_1']  = row.get('OT_lag_1', 0)  + SHIFT_MAGNITUDE
        row['OT_lag_24'] = row.get('OT_lag_24', 0) + SHIFT_MAGNITUDE

    y_true = row[TARGET_COL]
    x_df   = pd.DataFrame([{k: row[k] for k in FEATURE_COLS}])
    y_pred = float(champion.predict(x_df)[0])

    synth_actuals.append(y_true)
    synth_preds.append(y_pred)
    synth_residuals.append(y_true - y_pred)

    event = synth_monitor.update(y_pred=y_pred, y_true=y_true)
    if event:
        synth_events.append(event)

print(f'Synthetic simulation: {synth_monitor.drift_count} drift events')
for ev in synth_events[:10]:  # show first 10
    print(f'  row {ev.row_index:4d} | {ev.severity:4s} | {ev.triggered_detectors}')

# Verify detection near the shift
high_events = [ev for ev in synth_events if ev.severity == 'high']
if high_events:
    first_high = high_events[0]
    lag = first_high.row_index - SHIFT_START
    print(f'\nFirst HIGH-severity event at row {first_high.row_index} '
          f'({lag} rows after shift at {SHIFT_START})')

## 6. Synthetic Drift Plot

In [ ]:
fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['OT Forecast (synthetic drift)', 'Residuals'])

fig3.add_trace(go.Scatter(y=synth_actuals, name='Actual',    line=dict(color='steelblue')), row=1, col=1)
fig3.add_trace(go.Scatter(y=synth_preds,   name='Predicted', line=dict(color='orange', dash='dot')), row=1, col=1)
fig3.add_trace(go.Scatter(y=synth_residuals, name='Residual', line=dict(color='grey'), opacity=0.7), row=2, col=1)
fig3.add_hline(y=0, line_dash='dot', row=2, col=1)

# Shift injection marker
for row_num in [1, 2]:
    fig3.add_vline(x=SHIFT_START, line_color='blue', line_dash='longdash',
                   annotation_text='shift injected', row=row_num, col=1)

for ev in synth_events:
    color = 'red' if ev.severity == 'high' else 'coral'
    for row_num in [1, 2]:
        fig3.add_vline(x=ev.row_index, line_color=color, line_dash='dash', row=row_num, col=1)

fig3.update_layout(title='Synthetic Drift Injection Results', height=600, hovermode='x unified')
fig3.show()

## 7. Retrainer Demo

Take the first high-severity event from the synthetic simulation and invoke the retrainer.
Because no champion is registered in this notebook context, the promotion will record
`champion_rmse=None` (inf internally), so any finite challenger RMSE triggers promotion.

In [ ]:
from src.drift.retrainer import DriftRetrainer

demo_retrainer = DriftRetrainer(val_df=val_df, model_name=MODEL_NAME)

# Feed rows into the buffer up to the first high-severity event
synth_high = [ev for ev in synth_events if ev.severity == 'high']
if not synth_high:
    print('No high-severity event in synthetic stream — try increasing SHIFT_MAGNITUDE.')
else:
    first_high_event = synth_high[0]
    print(f'Triggering retrainer on event at row {first_high_event.row_index}')

    for i, row in enumerate(stream_test_data(delay_seconds=0.0)):
        demo_retrainer.ingest(dict(row))
        if i >= first_high_event.row_index:
            break

    result = demo_retrainer.handle(first_high_event)
    print(f'\nPromotionResult:')
    print(f'  promoted        : {result.promoted}')
    print(f'  challenger_rmse : {result.challenger_rmse:.4f}')
    print(f'  champion_rmse   : {result.champion_rmse}')
    print(f'  mode            : {result.mode}')
    print(f'  run_id          : {result.run_id}')